In [1]:
import pyterrier as pt
import requests
import pandas as pd
import pickle
import nltk
import numpy as np
import json

In [2]:
url_base = "https://www.bibsonomy.org/json/search/"
url_attr = "?items=1000"

def df_from_query(query):
    url = url_base + query.replace(" ", "%20") + url_attr  
    result = requests.get(url)
    json_data = result.json()
    dataframe = pd.DataFrame(json_data["items"])
    dataframe["query"] = query
    return dataframe

dataframes = []
queries = ["database", "text", "library", "archive"]

for q in queries:
    df = df_from_query(q)
    dataframes.append(df)
    print(f"{len(df)} Ergebnisse für query {q} gefunden.")

data = pd.concat(dataframes, ignore_index=True)

2000 Ergebnisse für query database gefunden.
2000 Ergebnisse für query text gefunden.
2000 Ergebnisse für query library gefunden.
2000 Ergebnisse für query archive gefunden.


In [5]:
data

,type,id,tags,intraHash,label,user,description,date,changeDate,count,...,attachments,categories,url_pdf,data,online,titleaddon,refnum,in-text-citation,w-type,w-projects
0,Bookmark,https://www.bibsonomy.org/url/cc45493bf442b78d...,"[web2.0, database, desktop, aggregator, daily,...",cc45493bf442b78d923b0b1b9342676b,Office 2.0 Database,springnet,Office 2.0 Database,2006-04-19 19:49:42,2008-06-29 17:52:49,13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Bookmark,https://www.bibsonomy.org/url/61474ed79d5c3f63...,"[tool, database, 3403, Error, Fix,, recovery,,...",61474ed79d5c3f63c1d0b6dcd8c653f5,SQL Database 3403 Error Fix,traktools,SysTools Software releases an effective SQL da...,2009-09-15 19:13:52,2009-09-15 19:13:52,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Bookmark,https://www.bibsonomy.org/url/3cd373dd8ed9c534...,"[Article(s), Design, Database, Information]",3cd373dd8ed9c53483d39eaa928686f6,Database Sharding,panic,"The term ""sharding"" was coined by Google engin...",2011-10-01 02:24:14,2011-10-01 02:27:38,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Bookmark,https://www.bibsonomy.org/url/cc45493bf442b78d...,"[web2.0, 2007, database, socialsoftware, tools...",cc45493bf442b78d923b0b1b9342676b,Office 2.0 Database,akpe,"Overview of roughly 250 Web 2.0 applications, ...",2007-03-14 20:26:50,2011-03-14 16:42:58,13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Bookmark,https://www.bibsonomy.org/url/2f5f76391c720a9b...,[databases],2f5f76391c720a9b688f60ec6f900f1d,Encyclopedia of Database Systems,ludaesch,Encyclopedia of Database Systems,2007-02-08 11:20:08,2007-09-26 06:23:20,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7995,Publication,https://www.bibsonomy.org/bibtex/2040f345d94cd...,[dblp],040f345d94cd5d5e6c4e6a2ca7d2f43e,Faculty self-archiving behavior: Factors affec...,dblp,,2020-06-24 00:00:00,2024-04-10 00:42:34,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7996,Publication,https://www.bibsonomy.org/bibtex/28217f4fe9f6a...,[dblp],8217f4fe9f6a3929c0d365d35447be84,Sensitivity and Access: Unlocking the Colonial...,dblp,,2024-08-08 00:00:00,2024-08-12 07:04:36,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7997,Publication,https://www.bibsonomy.org/bibtex/29994ea45725f...,[dblp],9994ea45725f4b521a1ca2da5f6b1bae,Online hot topic detection from web news archi...,dblp,,2021-09-19 00:00:00,2024-04-10 19:24:49,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7998,Publication,https://www.bibsonomy.org/bibtex/2ee55445bc456...,[dblp],ee55445bc456b80407d888fe2948f6a1,Serge Modular Archive Instrument (SMAI): Bridg...,dblp,,2024-09-16 00:00:00,2024-09-23 07:12:24,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
#drop columns where at least 50% of the rows have missing values
df_filtered = data.dropna(axis=1, thresh=len(df)*0.5)

In [9]:
df_filtered.columns

Index(['type', 'id', 'tags', 'intraHash', 'label', 'user', 'description',
       'date', 'changeDate', 'count', 'url', 'interHash', 'pub-type',
       'journal', 'year', 'author', 'authors', 'volume', 'abstract',
       'bibtexKey', 'pages', 'publisher', 'number', 'booktitle', 'ee',
       'query'],
      dtype='object')

In [11]:
useful_columns = ['id', 'tags', 'label', 'description', 'date', 'changeDate', 'url', 'pub-type', 'year', 'author', 'authors']
df_filtered = df_filtered[useful_columns]

In [13]:
df_filtered

,id,tags,label,description,date,changeDate,url,pub-type,year,author,authors
0,https://www.bibsonomy.org/url/cc45493bf442b78d...,"[web2.0, database, desktop, aggregator, daily,...",Office 2.0 Database,Office 2.0 Database,2006-04-19 19:49:42,2008-06-29 17:52:49,http://itredux.com/office-20/database/,NaN,NaN,NaN,NaN
1,https://www.bibsonomy.org/url/61474ed79d5c3f63...,"[tool, database, 3403, Error, Fix,, recovery,,...",SQL Database 3403 Error Fix,SysTools Software releases an effective SQL da...,2009-09-15 19:13:52,2009-09-15 19:13:52,http://www.sqlrecoverytool.com/sql-database-34...,NaN,NaN,NaN,NaN
2,https://www.bibsonomy.org/url/3cd373dd8ed9c534...,"[Article(s), Design, Database, Information]",Database Sharding,"The term ""sharding"" was coined by Google engin...",2011-10-01 02:24:14,2011-10-01 02:27:38,http://www.codefutures.com/database-sharding/,NaN,NaN,NaN,NaN
3,https://www.bibsonomy.org/url/cc45493bf442b78d...,"[web2.0, 2007, database, socialsoftware, tools...",Office 2.0 Database,"Overview of roughly 250 Web 2.0 applications, ...",2007-03-14 20:26:50,2011-03-14 16:42:58,http://itredux.com/office-20/database/,NaN,NaN,NaN,NaN
4,https://www.bibsonomy.org/url/2f5f76391c720a9b...,[databases],Encyclopedia of Database Systems,Encyclopedia of Database Systems,2007-02-08 11:20:08,2007-09-26 06:23:20,http://refworks.springer.com/database-systems/,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
7995,https://www.bibsonomy.org/bibtex/2040f345d94cd...,[dblp],Faculty self-archiving behavior: Factors affec...,,2020-06-24 00:00:00,2024-04-10 00:42:34,http://dblp.uni-trier.de/db/conf/asist/asist20...,inproceedings,2007,[Jihyun Kim],"[{'first': 'Jihyun', 'last': 'Kim'}]"
7996,https://www.bibsonomy.org/bibtex/28217f4fe9f6a...,[dblp],Sensitivity and Access: Unlocking the Colonial...,,2024-08-08 00:00:00,2024-08-12 07:04:36,http://dblp.uni-trier.de/db/journals/dhq/dhq18...,article,2024,"[Jonathan Dentler, Lise Jaillant, Daniel Folia...","[{'first': 'Jonathan', 'last': 'Dentler'}, {'f..."
7997,https://www.bibsonomy.org/bibtex/29994ea45725f...,[dblp],Online hot topic detection from web news archi...,,2021-09-19 00:00:00,2024-04-10 19:24:49,http://dblp.uni-trier.de/db/conf/fskd/fskd2014...,inproceedings,2014,"[RuiGuo Yu, Mankun Zhao, Peng Chang, MuWen He]","[{'first': 'RuiGuo', 'last': 'Yu'}, {'first': ..."
7998,https://www.bibsonomy.org/bibtex/2ee55445bc456...,[dblp],Serge Modular Archive Instrument (SMAI): Bridg...,,2024-09-16 00:00:00,2024-09-23 07:12:24,http://dblp.uni-trier.de/db/conf/nime/nime2023...,inproceedings,2023,"[Ted Moore, Jean Brazeau]","[{'first': 'Ted', 'last': 'Moore'}, {'first': ..."


In [15]:
#pyterrier exspects a docno field
df_filtered['docno'] = df_filtered['id']
df_filtered['text'] = df_filtered['label']

In [17]:
index_folder = "./df_index"

indexer = pt.DFIndexer(index_folder, overwrite=True)
index_ref = indexer.index(df_filtered['text'], df_filtered['docno'])

print(f"path to our index: {index_ref.toString()}")

/var/folders/88/9blsh45918g8y3n5vl6vztxr0000gq/T/ipykernel_3547/146072258.py:3: DeprecationWarning: Call to deprecated class DFIndexer. (use pt.terrier.IterDictIndexer().index(dataframe.to_dict(orient='records')) instead) -- Deprecated since version 0.11.0.
  indexer = pt.DFIndexer(index_folder, overwrite=True)
Java started (triggered by TerrierIndexer.__init__) and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


09:14:34.413 [main] WARN org.terrier.structures.indexing.Indexer -- Adding an empty document to the index (https://www.bibsonomy.org/url/45d7bb8eace077fe1f46f404529960f4/iblis) - further warnings are suppressed
09:14:35.454 [main] WARN org.terrier.structures.indexing.Indexer -- Indexed 4 empty documents
path to our index: ./df_index/data.properties


In [19]:
index = pt.IndexFactory.of(index_ref)

In [21]:
print(index.getCollectionStatistics().toString())

Number of documents: 8000
Number of terms: 8481
Number of postings: 46839
Number of fields: 0
Number of tokens: 48611
Field names: []
Positions:   false



In [23]:
for kv in index.getLexicon():
    print(f"{kv.getKey()} -> {kv.getValue().toString()}")

0 -> term1 Nt=94 TF=103 maxTF=3 @{0 0 0}
000 -> term4818 Nt=5 TF=5 maxTF=1 @{0 132 1}
0002 -> term4089 Nt=4 TF=4 maxTF=1 @{0 145 3}
01 -> term5346 Nt=3 TF=3 maxTF=1 @{0 150 1}
017 -> term5389 Nt=1 TF=1 maxTF=1 @{0 158 3}
02 -> term3559 Nt=4 TF=4 maxTF=1 @{0 161 5}
03 -> term3626 Nt=3 TF=3 maxTF=1 @{0 169 7}
0301 -> term3463 Nt=1 TF=1 maxTF=1 @{0 177 1}
04 -> term3599 Nt=3 TF=3 maxTF=1 @{0 180 1}
05 -> term1331 Nt=3 TF=3 maxTF=1 @{0 187 5}
050 -> term2507 Nt=1 TF=1 maxTF=1 @{0 195 3}
06 -> term4354 Nt=3 TF=3 maxTF=1 @{0 198 1}
062 -> term7186 Nt=1 TF=1 maxTF=1 @{0 206 7}
07 -> term3868 Nt=3 TF=3 maxTF=1 @{0 210 1}
08 -> term3969 Nt=3 TF=4 maxTF=2 @{0 217 5}
09 -> term5266 Nt=1 TF=1 maxTF=1 @{0 224 0}
1 -> term767 Nt=48 TF=58 maxTF=4 @{0 227 2}
10 -> term1029 Nt=24 TF=25 maxTF=2 @{0 310 2}
100 -> term3368 Nt=8 TF=8 maxTF=1 @{0 355 1}
101 -> term3875 Nt=1 TF=1 maxTF=1 @{0 374 5}
1039 -> term5958 Nt=1 TF=1 maxTF=1 @{0 377 5}
10g -> term96 Nt=7 TF=7 maxTF=1 @{0 380 7}
11 -> term1275 Nt=11 T

In [25]:
term_freq_dict = {}

for kv in index.getLexicon():
    term_freq_dict[kv.getKey()] = kv.getValue().frequency

In [27]:
term_freq_dict_sorted = sorted(term_freq_dict.items(), key=lambda x: x[1], reverse=True)

In [29]:
top_k = 30
term_freq_dict_sorted[:top_k]

[('librari', 2267),
 ('text', 2092),
 ('archiv', 1875),
 ('databas', 1837),
 ('data', 361),
 ('base', 320),
 ('web', 320),
 ('digit', 315),
 ('system', 314),
 ('us', 257),
 ('mine', 243),
 ('new', 236),
 ('open', 235),
 ('internet', 234),
 ('analysi', 202),
 ('search', 191),
 ('research', 185),
 ('classif', 180),
 ('model', 175),
 ('servic', 174),
 ('onlin', 163),
 ('manag', 159),
 ('learn', 158),
 ('org', 156),
 ('gener', 153),
 ('tool', 134),
 ('semant', 133),
 ('public', 132),
 ('scienc', 131),
 ('free', 130)]

In [31]:
search_engine = pt.BatchRetrieve(index, wmodel="Tf")

/var/folders/88/9blsh45918g8y3n5vl6vztxr0000gq/T/ipykernel_3547/951663741.py:1: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  search_engine = pt.BatchRetrieve(index, wmodel="Tf")


In [33]:
search_engine.search("system")

,qid,docid,docno,rank,score,query
0,1,750,https://www.bibsonomy.org/url/be868d201af37d4a...,0,2.0,system
1,1,1845,https://www.bibsonomy.org/bibtex/27cebe02af1e4...,1,2.0,system
2,1,1861,https://www.bibsonomy.org/bibtex/2e8737a4cce47...,2,2.0,system
3,1,1863,https://www.bibsonomy.org/bibtex/2072b030962e3...,3,2.0,system
4,1,3853,https://www.bibsonomy.org/bibtex/280f75bb46fcb...,4,2.0,system
...,...,...,...,...,...,...
303,1,7950,https://www.bibsonomy.org/bibtex/2ae6a916923ad...,303,1.0,system
304,1,7968,https://www.bibsonomy.org/bibtex/2390b02f23f3d...,304,1.0,system
305,1,7973,https://www.bibsonomy.org/bibtex/25d3f12c5e0f6...,305,1.0,system
306,1,7974,https://www.bibsonomy.org/bibtex/270a5cfbabcbb...,306,1.0,system


In [35]:
from nltk.stem import PorterStemmer
stemmer = PorterStemmer()

In [37]:
stem = stemmer.stem("learning")
stem

'learn'

In [39]:
lexicon = index.getLexicon()
lexicon[stem].getDocumentFrequency()

155

In [41]:
index.getCollectionStatistics().numberOfDocuments

8000

In [43]:
def get_idf_for_term(term, index):
  lex = index.getLexicon()
  stemmed_term = stemmer.stem(term)

  if not stemmed_term in lex:
    return 
    
  lex_entry = lex[stemmed_term]

  df_term = lex_entry.getDocumentFrequency()
  N = index.getCollectionStatistics().numberOfDocuments
  
  #inverse document frequency
  idf = N/df_term

  #apply logarithm(base 10) to idf
  log_idf = np.emath.logn(10, idf)

  return log_idf

In [45]:
get_idf_for_term("learning", index)

1.712758288821652

In [49]:
#define your own tf_idf method here:

def calc_tf_idf(query, docno, index):
    #remember that tfidf is the product of two components
    #hint: the tf model search result contains tf frequencies
    return 0

In [47]:
search_engine_tfidf = pt.BatchRetrieve(index, wmodel="TF_IDF")

/var/folders/88/9blsh45918g8y3n5vl6vztxr0000gq/T/ipykernel_3547/2880449899.py:1: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  search_engine_tfidf = pt.BatchRetrieve(index, wmodel="TF_IDF")


In [51]:
search_engine_tfidf.search("database")

,qid,docid,docno,rank,score,query
0,1,751,https://www.bibsonomy.org/url/a096f0b12753c22e...,0,2.170079,database
1,1,281,https://www.bibsonomy.org/url/b05f7889d6a0731c...,1,2.058956,database
2,1,287,https://www.bibsonomy.org/url/b05f7889d6a0731c...,2,2.058956,database
3,1,1007,https://www.bibsonomy.org/bibtex/2b8151c596a8d...,3,2.058956,database
4,1,1785,https://www.bibsonomy.org/bibtex/25802e079a691...,4,2.056275,database
...,...,...,...,...,...,...
995,1,573,https://www.bibsonomy.org/url/262cdca8951e45ec...,995,1.360506,database
996,1,579,https://www.bibsonomy.org/url/cd8d11db33dc293f...,996,1.360506,database
997,1,580,https://www.bibsonomy.org/url/61ccc5248c3559d5...,997,1.360506,database
998,1,581,https://www.bibsonomy.org/url/17e00df720be1825...,998,1.360506,database


In [53]:
# Index with multiple fields
df_dict = df_filtered.to_dict(orient='records')

In [55]:
df_dict

[{'id': 'https://www.bibsonomy.org/url/cc45493bf442b78d923b0b1b9342676b/springnet',
  'tags': ['web2.0',
   'database',
   'desktop',
   'aggregator',
   'daily',
   'incredible',
   'applications',
   'bookmarking',
   'blog'],
  'label': 'Office 2.0 Database',
  'description': 'Office 2.0 Database',
  'date': '2006-04-19 19:49:42',
  'changeDate': '2008-06-29 17:52:49',
  'url': 'http://itredux.com/office-20/database/',
  'pub-type': nan,
  'year': nan,
  'author': nan,
  'authors': nan,
  'docno': 'https://www.bibsonomy.org/url/cc45493bf442b78d923b0b1b9342676b/springnet',
  'text': 'Office 2.0 Database'},
 {'id': 'https://www.bibsonomy.org/url/61474ed79d5c3f63c1d0b6dcd8c653f5/traktools',
  'tags': ['tool',
   'database',
   '3403',
   'Error',
   'Fix,',
   'recovery,',
   'Database',
   'Recovery',
   'SQL'],
  'label': 'SQL Database 3403 Error Fix',
  'description': 'SysTools Software releases an effective SQL database recovery software to solve your sql database problems as sql d

In [63]:
index_folder_mult = "./index_mult"

fields=['docno', 'text', 'tags', 'description']

indexer_mult = pt.IterDictIndexer(index_folder_mult, meta={'docno': 200, 'text': 4096}, overwrite=True)
index_ref_mult = indexer_mult.index(df_dict)

print(f"path to our index: {index_ref_mult.toString()}")

09:18:16.597 [ForkJoinPool-1-worker-1] WARN org.terrier.structures.indexing.Indexer -- Adding an empty document to the index (https://www.bibsonomy.org/url/45d7bb8eace077fe1f46f404529960f4/iblis) - further warnings are suppressed
09:18:16.975 [ForkJoinPool-1-worker-1] WARN org.terrier.structures.indexing.Indexer -- Indexed 4 empty documents
path to our index: ./index_mult/data.properties


In [65]:
index_mult = pt.IndexFactory.of(index_ref_mult)

In [67]:
print(index_mult.getCollectionStatistics().toString())


Number of documents: 8000
Number of terms: 8495
Number of postings: 46900
Number of fields: 1
Number of tokens: 48683
Field names: [text]
Positions:   false



In [69]:
search_engine_mult = pt.terrier.Retriever(index_mult, wmodel="TF_IDF")

In [71]:
res = search_engine_mult.search("intelligence")
res

,qid,docid,docno,rank,score,query
0,1,119,https://www.bibsonomy.org/url/8f1fb00012322e81...,0,5.319447,intelligence
1,1,3009,https://www.bibsonomy.org/bibtex/21a61a34d4984...,1,5.319447,intelligence
2,1,3092,https://www.bibsonomy.org/bibtex/21a61a34d4984...,2,5.319447,intelligence
3,1,3667,https://www.bibsonomy.org/bibtex/2508ce3b4cfd8...,3,5.319447,intelligence
4,1,6244,https://www.bibsonomy.org/url/7e3b188d3277ff0c...,4,5.319447,intelligence
5,1,3407,https://www.bibsonomy.org/bibtex/2326beb667d4c...,5,4.933700,intelligence
6,1,3545,https://www.bibsonomy.org/bibtex/255165fc7f1f9...,6,4.933700,intelligence
7,1,573,https://www.bibsonomy.org/url/262cdca8951e45ec...,7,4.600116,intelligence
8,1,2796,https://www.bibsonomy.org/url/e94f2e8c95d6fd01...,8,4.600116,intelligence
9,1,3205,https://www.bibsonomy.org/bibtex/2934ad4d54596...,9,4.600116,intelligence


In [75]:
#save our data
pickle.dump(df_filtered, open("index_mult/df_publications.pkl", "wb"))

In [77]:
#install steamlit
%pip install streamlit

Note: you may need to restart the kernel to use updated packages.
